# Dataset preparation
The goal if to take the merged paragraphs and labels and quantify them.

In [1]:
import pandas as pd
import numpy as np

In [2]:
from sklearn.model_selection import StratifiedGroupKFold

In [3]:
p_df = pd.read_parquet("legal_paragraphs.parquet")

In [4]:
p_df.head()

,document_id,paragraph_index,paragraph_order,numbered_text,text,section
0,116096838,1,1,[paragraph_id=1] ХАРКІВСЬКИЙ АПЕЛЯЦІЙНИЙ СУД,ХАРКІВСЬКИЙ АПЕЛЯЦІЙНИЙ СУД,introductory
1,116096838,2,2,[paragraph_id=2] _____________________________...,______________________________________________...,introductory
2,116096838,3,3,[paragraph_id=3] Справа № 818/1367т/23,Справа № 818/1367т/23,introductory
3,116096838,4,4,[paragraph_id=4] Провадження № 11кс/818/1321/23,Провадження № 11кс/818/1321/23,introductory
4,116096838,5,5,[paragraph_id=5] УХВАЛА,УХВАЛА,introductory


In [5]:
p_df.shape, p_df.columns

((289367, 6),
 Index(['document_id', 'paragraph_index', 'paragraph_order', 'numbered_text',
        'text', 'section'],
       dtype='str'))

In [8]:
p_df["section"].unique().tolist()

['introductory', 'reasoning', 'operative']

In [13]:
LABELS_TO_IDS = {
    value: key for key, value in enumerate(p_df["section"].unique().tolist())
}

In [14]:
LABELS_TO_IDS

{'introductory': 0, 'reasoning': 1, 'operative': 2}

In [15]:
p_df["document_id"] = p_df["document_id"].astype(str)
p_df = p_df.sort_values(["document_id", "paragraph_index"], kind="stable").reset_index(drop=True)

In [16]:
assert not p_df.duplicated(["document_id", "paragraph_index"]).any()
assert set(p_df["section"].dropna()).issubset(LABELS_TO_IDS)

In [17]:
by_document = p_df.groupby("document_id")["text"]
previous_1 = by_document.shift(1).fillna("")
previous_2 = by_document.shift(2).fillna("")

In [18]:
p_df["context"] = (previous_2 + "\n\n" + previous_1).str.strip()
p_df["label"] = p_df["section"].map(LABELS_TO_IDS).astype("int8")

In [19]:
p_df.head()

,document_id,paragraph_index,paragraph_order,numbered_text,text,section,context,label
0,116061850,1,1,[paragraph_id=1] Справа № 127/30543/23,Справа № 127/30543/23,introductory,,0
1,116061850,2,2,[paragraph_id=2] Провадження № 1-кп/127/904/23,Провадження № 1-кп/127/904/23,introductory,Справа № 127/30543/23,0
2,116061850,3,3,[paragraph_id=3] В И Р О К,В И Р О К,introductory,Справа № 127/30543/23\n\nПровадження № 1-кп/12...,0
3,116061850,4,4,[paragraph_id=4] ІМЕНЕМ УКРАЇНИ,ІМЕНЕМ УКРАЇНИ,introductory,Провадження № 1-кп/127/904/23\n\nВ И Р О К,0
4,116061850,5,5,[paragraph_id=5] 28.12.2023 |місто Вінниця|,28.12.2023 |місто Вінниця|,introductory,В И Р О К\n\nІМЕНЕМ УКРАЇНИ,0


Split dataset
80-10-10

In [20]:
folds = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)

In [24]:
p_df["fold"] = -1
for fold, (_, held_out_indicies) in enumerate(
    folds.split(p_df, y=p_df["label"], groups=p_df["document_id"])
):
    p_df.loc[held_out_indicies, "fold"] = fold

In [29]:
p_df.head()

,document_id,paragraph_index,paragraph_order,numbered_text,text,section,context,label,fold
0,116061850,1,1,[paragraph_id=1] Справа № 127/30543/23,Справа № 127/30543/23,introductory,,0,4
1,116061850,2,2,[paragraph_id=2] Провадження № 1-кп/127/904/23,Провадження № 1-кп/127/904/23,introductory,Справа № 127/30543/23,0,4
2,116061850,3,3,[paragraph_id=3] В И Р О К,В И Р О К,introductory,Справа № 127/30543/23\n\nПровадження № 1-кп/12...,0,4
3,116061850,4,4,[paragraph_id=4] ІМЕНЕМ УКРАЇНИ,ІМЕНЕМ УКРАЇНИ,introductory,Провадження № 1-кп/127/904/23\n\nВ И Р О К,0,4
4,116061850,5,5,[paragraph_id=5] 28.12.2023 |місто Вінниця|,28.12.2023 |місто Вінниця|,introductory,В И Р О К\n\nІМЕНЕМ УКРАЇНИ,0,4


In [32]:
p_df["fold"].unique()

array([4, 6, 8, 2, 9, 3, 0, 1, 5, 7])

In [34]:
p_df["split"] = "train"
p_df.loc[p_df["fold"] == 0, "split"] = "test"
p_df.loc[p_df["fold"] == 1, "split"] = "validation"

In [35]:
training_df = p_df[[
    "document_id",
    "paragraph_index",
    "context",
    "text",
    "section",
    "label",
    "split",
]].copy()

In [36]:
print(training_df["split"].value_counts())

split
train         231493
test           28945
validation     28929
Name: count, dtype: int64


In [37]:
pd.crosstab(training_df["split"], training_df["section"])

section,introductory,operative,reasoning
split,,,
test,8643,6574,13728
train,69126,52615,109752
validation,8642,6576,13711


In [38]:
training_df.head()

,document_id,paragraph_index,context,text,section,label,split
0,116061850,1,,Справа № 127/30543/23,introductory,0,train
1,116061850,2,Справа № 127/30543/23,Провадження № 1-кп/127/904/23,introductory,0,train
2,116061850,3,Справа № 127/30543/23\n\nПровадження № 1-кп/12...,В И Р О К,introductory,0,train
3,116061850,4,Провадження № 1-кп/127/904/23\n\nВ И Р О К,ІМЕНЕМ УКРАЇНИ,introductory,0,train
4,116061850,5,В И Р О К\n\nІМЕНЕМ УКРАЇНИ,28.12.2023 |місто Вінниця|,introductory,0,train


In [39]:
training_df.to_parquet("legal_paragraphs_dataset.parquet", index=False)

In [40]:
%reset